# Movie Expert Agent

OpenAI Agents SDK + Clean Architecture 기반 영화 추천 에이전트

## Architecture
```
domain/          - 엔티티, 리포지토리 인터페이스 (비즈니스 규칙)
application/     - Use Cases, Context (유스케이스 조율)
infrastructure/  - API 리포지토리 구현 (외부 시스템)
presentation/    - Agent, Tools 정의 (사용자 인터페이스)
```

## 1. 환경 설정

In [15]:
import os
from dotenv import load_dotenv
from agents import RunConfig
from agents.models.openai_provider import OpenAIProvider

load_dotenv()

# RunConfig 설정 (커스텀 base_url 지원)
base_url = os.getenv("AI_BASE_URL")
if base_url:
    run_config = RunConfig(
        model_provider=OpenAIProvider(
            base_url=base_url,
            use_responses=False,
        ),
        tracing_disabled=True,
    )
else:
    run_config = RunConfig()

print("환경 설정 완료")

환경 설정 완료


## 2. DI Composition Root - 의존성 조립

In [16]:
from application.context import AppContext
from application.use_cases import (
    GetMovieCreditsUseCase,
    GetMovieDetailsUseCase,
    GetPopularMoviesUseCase,
    GetSimilarMoviesUseCase,
)
from infrastructure.api_movie_repository import ApiMovieRepository

# Infrastructure -> Application -> Presentation 순으로 조립
repo = ApiMovieRepository()
context = AppContext(
    get_popular_movies=GetPopularMoviesUseCase(repo),
    get_movie_details=GetMovieDetailsUseCase(repo),
    get_movie_credits=GetMovieCreditsUseCase(repo),
    get_similar_movies=GetSimilarMoviesUseCase(repo),
)

print("의존성 조립 완료")

의존성 조립 완료


## 3. Agent 생성

In [17]:
from presentation.agent import create_agent

agent = create_agent()
print(f"Agent: {agent.name}")
print(f"Tools: {[t.name for t in agent.tools]}")

Agent: Movie Expert
Tools: ['get_popular_movies', 'get_movie_details', 'get_movie_credits', 'get_similar_movies']


## 4. 대화 실행

아래 셀에서 `user_input`을 변경하며 에이전트와 대화할 수 있습니다.
`input_items` 리스트가 대화 히스토리를 유지합니다.

In [18]:
# 대화 히스토리 초기화
input_items = []

In [19]:
from agents import Runner

user_input = input("질문을 입력하세요: ")

input_items.append({"role": "user", "content": user_input})
result = await Runner.run(agent, input_items, context=context, run_config=run_config)
input_items = result.to_input_list()

print(f"\nAgent: {result.final_output}")

  [get_popular_movies() 호출 중...]
  [get_popular_movies() 완료]

Agent: 현재 인기 있는 영화 목록은 다음과 같습니다:

1. **Your Heart Will Be Broken** (ID: 1523145) - 평점: 6.9, 개봉일: 2026-03-26
2. **Avatar: Fire and Ash** (ID: 83533) - 평점: 7.373, 개봉일: 2025-12-17
3. **Hoppers** (ID: 1327819) - 평점: 7.6, 개봉일: 2026-03-04
4. **The Super Mario Bros. Movie** (ID: 502356) - 평점: 7.588, 개봉일: 2023-04-05
5. **The Super Mario Galaxy Movie** (ID: 1226863) - 평점: 6.888, 개봉일: 2026-04-01
6. **Shelter** (ID: 1290821) - 평점: 6.784, 개봉일: 2026-01-28
7. **Project Hail Mary** (ID: 687163) - 평점: 8.224, 개봉일: 2026-03-15
8. **Thrash** (ID: 1290417) - 평점: 5.9, 개봉일: 2026-04-10
9. **Crime 101** (ID: 1171145) - 평점: 7.031, 개봉일: 2026-02-11
10. **Greenland 2: Migration** (ID: 840464) - 평점: 6.5, 개봉일: 2026-01-07

더 궁금한 영화가 있으신가요?


### 후속 질문 (이전 대화 컨텍스트 유지)

In [20]:
user_input = input("질문을 입력하세요: ")

input_items.append({"role": "user", "content": user_input})
result = await Runner.run(agent, input_items, context=context, run_config=run_config)
input_items = result.to_input_list()

print(f"\nAgent: {result.final_output}")

  [get_movie_details() 호출 중...]
  [get_movie_details() 완료]
  [get_movie_credits() 호출 중...]
  [get_movie_credits() 완료]

Agent: **아바타: 화염과 재** (Avatar: Fire and Ash)에 대한 자세한 정보입니다:

- **개봉일**: 2025-12-17
- **평점**: 7.373
- **장르**: 공상 과학, 모험, 판타지
- **러닝타임**: 198분
- **상태**: 개봉 예정
- **언어**: 영어
- **예산**: 약 3억 5천만 달러
- **수익**: 약 14억 8596만 달러
- **태그라인**: The world of Pandora will change forever.

**줄거리**: 
전투에서 큰 상처를 입은 RDA와의 전쟁 이후, Jake Sully와 Neytiri는 그들의 장남을 잃고 새로운 위협에 직면한다. 바로 잔인한 Varang이 이끄는 폭력적이고 권력을 탐하는 Na'vi 부족인 'Ash People'이다. Jake의 가족은 생존과 판도라의 미래를 위해 싸워야 하며, 그 과정에서 감정적 및 신체적 한계에 도전하게 된다.

**출연진**:
- Sam Worthington (Jake)
- Zoe Saldaña (Neytiri)
- Sigourney Weaver (Kiri)
- Stephen Lang (Quaritch)
- Oona Chaplin (Varang)
- Jack Champion (Spider)
- Kate Winslet (Ronal)
- Cliff Curtis (Tonowari)
- Joel David Moore (Norm)
- CCH Pounder (Mo'at)

더 궁금한 점이 있으면 말씀해 주세요!


In [21]:
user_input = input("질문을 입력하세요: ")

input_items.append({"role": "user", "content": user_input})
result = await Runner.run(agent, input_items, context=context, run_config=run_config)
input_items = result.to_input_list()

print(f"\nAgent: {result.final_output}")

  [get_similar_movies() 호출 중...]
  [get_similar_movies() 완료]

Agent: **아바타: 화염과 재**와 유사한 영화 추천 목록입니다:

1. **The Faculty** (ID: 9276) - 평점: 6.563, 개봉일: 1998-12-25
2. **Species** (ID: 9348) - 평점: 5.976, 개봉일: 1995-07-07
3. **Mad Max Beyond Thunderdome** (ID: 9355) - 평점: 6.219, 개봉일: 1985-06-29
4. **God Bless Dancougar** (ID: 259907) - 평점: 6, 개봉일: 1987-03-19
5. **Honour** (ID: 605592) - 평점: 7.7, 개봉일: 2018-11-06
6. **Space Case** (ID: 783661) - 평점: 0, 개봉일: 2021-01-01
7. **The Crow: Salvation** (ID: 9456) - 평점: 5.5, 개봉일: 2000-01-23
8. **Garfield: A Tail of Two Kitties** (ID: 9513) - 평점: 5.613, 개봉일: 2006-06-15
9. **Superman III** (ID: 9531) - 평점: 5.5, 개봉일: 1983-06-17
10. **Long Distance** (ID: 605722) - 평점: 6.4, 개봉일: 2024-07-12
11. **Spy Kids 2: The Island of Lost Dreams** (ID: 9488) - 평점: 5.7, 개봉일: 2002-08-07
12. **On Deadly Ground** (ID: 9624) - 평점: 5.2, 개봉일: 1994-02-18
13. **The Blob** (ID: 9599) - 평점: 6.84, 개봉일: 1988-08-05
14. **The Swan Princess: Kingdom of Music** (ID: 605829) - 평점: 5.7,